In [1]:
import os
import pandas as pd
import re, string, nltk
from datasets import load_dataset
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk import word_tokenize, pos_tag
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
import time

c:\Users\ysnxlmted\Projects\documents_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Создание папки для nltk данных, если её нет
nltk_data_dir = os.path.expanduser('../nltk_data')
if not os.path.exists(nltk_data_dir):
    os.makedirs(nltk_data_dir)

# Добавление пути в nltk
nltk.data.path.append(nltk_data_dir)

# Загрузка всех необходимых ресурсов
resources = ['punkt', 'wordnet', 'omw-1.4', 'punkt_tab', 
             'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords']

for resource in resources:
    try:
        nltk.download(resource, download_dir=nltk_data_dir, quiet=False)
    except:
        print(f"Ресурс {resource} уже загружен или произошла ошибка")

[nltk_data] Downloading package punkt to ../nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to ../nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to ../nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to ../nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     ../nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to ../nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[{}[\]()<>]', '', text)
    text = re.sub(r'\d+\.\d+\.\d+\.\d+', '', text)
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    text = re.sub(email_pattern, '', text)
    text = re.sub(r'\S+/\S+/\S+', '', text)
    text = re.sub(r'\S+\.\S+/\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[`"\']{2,}', ' ', text)
    text = re.sub(r'[`"\']', ' ', text)
    text = re.sub(r'[\[\]{}()<>]', ' ', text)
    text = re.sub(r'(\w+)-(\d+)', r'\1\2', text)
    text = re.sub(r'(\d+)-(\w+)', r'\1\2', text)
    smile_pattern = r'[:;=][\-^]?[)D\(\[\]pP]+'
    text = re.sub(smile_pattern, '', text)
    text = re.sub(r'([!?.,])\1+', r'\1', text)
    text = re.sub(r'-{2,}', ' ', text)
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', '', text)
    text = re.sub(r'[^\w\s]', ' ', text) 
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

stop_words = set(stopwords.words("english"))
additional_stops = {
    'would', 'could', 'should', 'might', 'may', 'get', 'go', 'see', 
    'know', 'like', 'want', 'need', 'say', 'think', 'come', 'take',
    'use', 'make', 'well', 'also', 'even', 'many', 'much', 'still',
    'however', 'though', 'although', 'since', 'yet', 'already'
}
stop_words.update(additional_stops)

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

lemmatizer = nltk.WordNetLemmatizer()

def preprocess_with_lemmatization(text, pos_filter="all"):
    tokens = word_tokenize(clean_text(text))
    tagged = pos_tag(tokens)
    lemmatized_tokens = []
    for token, tag in tagged:
        if token in stop_words or token in string.punctuation:
            continue
        is_noun_adj = tag.startswith('N') or tag.startswith('J')
        if pos_filter == "nouns_adj" and not is_noun_adj:
            continue
        lemmatized_tokens.append(lemmatizer.lemmatize(token, get_wordnet_pos(tag)))
    return " ".join(lemmatized_tokens)

stemmer = PorterStemmer()
def preprocess_with_stemming(text):
    tokens = word_tokenize(clean_text(text))
    res = []
    for token in tokens:
        if token in stop_words or token in string.punctuation: continue
        res.append(stemmer.stem(token))
    return " ".join(res)

In [2]:
from datasets import load_dataset
import pandas as pd

# 1. Загрузка
dataset = load_dataset("SetFit/20_newsgroups")
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# 2. Полный список категорий (порядок фиксирован для 20 newsgroups)
all_categories = [
    'alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 
    'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 
    'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 
    'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 
    'sci.space', 'soc.religion.christian', 'talk.politics.guns', 
    'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc'
]

# 3. Выбираем нужные 4 категории
selected_categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware", 
    "comp.graphics",
    "comp.windows.x"
]

# 4. Находим их ID (индексы в общем списке)
selected_ids = [all_categories.index(cat) for cat in selected_categories]

# 5. Фильтрация по ID
train_df = train_df[train_df['label'].isin(selected_ids)].reset_index(drop=True)
test_df = test_df[test_df['label'].isin(selected_ids)].reset_index(drop=True)

# 6. Перемаппинг лейблов в 0, 1, 2, 3
label_map = {old_id: new_id for new_id, old_id in enumerate(selected_ids)}
train_df['label'] = train_df['label'].map(label_map)
test_df['label'] = test_df['label'].map(label_map)

print(f"✅ Классы: {selected_categories}")
print(f"📊 Train: {len(train_df)}, Test: {len(test_df)}")

c:\Users\ysnxlmted\Projects\documents_classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [6]:
print("Предобработка данных (это может занять время)...")
train_df['text_simple'] = train_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'all'))
train_df['text_pos'] = train_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'nouns_adj'))
train_df['text_stem'] = train_df['text'].apply(preprocess_with_stemming)

test_df['text_simple'] = test_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'all'))
test_df['text_pos'] = test_df['text'].apply(lambda x: preprocess_with_lemmatization(x, 'nouns_adj'))
test_df['text_stem'] = test_df['text'].apply(preprocess_with_stemming)

Предобработка данных (это может занять время)...


In [7]:
param_grids = {
    "Decision Tree": {
        'clf__max_depth': [10, 20, None],
        'clf__min_samples_split': [2, 5]
    },
    "Random Forest": {
        'clf__n_estimators': [50, 100],
        'clf__max_depth': [10, 20, None]
    }
}

In [8]:
models_dict = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
}

In [9]:
lsa_configs = [
    {'name': 'No LSA', 'n_components': None},
    {'name': 'LSA-50', 'n_components': 50},
    {'name': 'LSA-100', 'n_components': 100},
    {'name': 'LSA-200', 'n_components': 200}
]

preprocessing_cols = [
    ('Lemmatization', 'text_simple'),
    ('Nouns+Adj', 'text_pos'),
    ('Stemming', 'text_stem')
]

results_all = []

In [10]:
for prep_name, col_name in preprocessing_cols:
    for lsa_conf in lsa_configs:
        # Векторизатор (всегда TF-IDF для LSA, так как он лучше работает с весами)
        # Если LSA нет, все равно TF-IDF обычно лучше Count для классификации
        
        for model_name, model in models_dict.items():
            # Сборка пайплайна
            steps = [('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2)))]
            
            if lsa_conf['n_components']:
                steps.append(('svd', TruncatedSVD(n_components=lsa_conf['n_components'], random_state=42)))
            
            steps.append(('clf', model))
            
            pipeline = Pipeline(steps)
            
            # Подготовка сетки для GridSearch
            # Важно: имена параметров в сетке должны совпадать с именами шагов пайплайна (clf__, svd__)
            base_grid = param_grids[model_name]
            
            # Если есть LSA, можно добавить настройку компонент в поиск, 
            # но в этом коде мы фиксируем LSA во внешнем цикле для чистоты эксперимента
            search_grid = base_grid

            grid_search = GridSearchCV(
                pipeline, 
                param_grid=search_grid, 
                cv=3, 
                scoring='f1_micro', 
                n_jobs=-1, 
                verbose=0
            )
            
            X_train = train_df[col_name]
            y_train = train_df['label']
            X_test = test_df[col_name]
            y_test = test_df['label']

            try:
                grid_search.fit(X_train, y_train)
                best_model = grid_search.best_estimator_
                y_pred = best_model.predict(X_test)
                
                f1_micro = f1_score(y_test, y_pred, average='micro')
                f1_macro = f1_score(y_test, y_pred, average='macro')
                
                results_all.append({
                    "Предобработка": prep_name,
                    "LSA": lsa_conf['name'],
                    "Модель": model_name,
                    "Best Params": grid_search.best_params_,
                    "F1 Micro": f1_micro,
                    "F1 Macro": f1_macro
                })
                print(f"Готово: {prep_name} | {lsa_conf['name']} | {model_name} | F1 Micro: {f1_micro:.4f}")
            except Exception as e:
                print(f"Ошибка в {model_name}: {e}")            

Готово: Lemmatization | No LSA | Decision Tree | F1 Micro: 0.5958
Готово: Lemmatization | No LSA | Random Forest | F1 Micro: 0.7450
Готово: Lemmatization | LSA-50 | Decision Tree | F1 Micro: 0.5964
Готово: Lemmatization | LSA-50 | Random Forest | F1 Micro: 0.7053
Готово: Lemmatization | LSA-100 | Decision Tree | F1 Micro: 0.6118
Готово: Lemmatization | LSA-100 | Random Forest | F1 Micro: 0.7111
Готово: Lemmatization | LSA-200 | Decision Tree | F1 Micro: 0.6054
Готово: Lemmatization | LSA-200 | Random Forest | F1 Micro: 0.7188
Готово: Nouns+Adj | No LSA | Decision Tree | F1 Micro: 0.6047
Готово: Nouns+Adj | No LSA | Random Forest | F1 Micro: 0.7271
Готово: Nouns+Adj | LSA-50 | Decision Tree | F1 Micro: 0.5874
Готово: Nouns+Adj | LSA-50 | Random Forest | F1 Micro: 0.6983
Готово: Nouns+Adj | LSA-100 | Decision Tree | F1 Micro: 0.5964
Готово: Nouns+Adj | LSA-100 | Random Forest | F1 Micro: 0.7040
Готово: Nouns+Adj | LSA-200 | Decision Tree | F1 Micro: 0.6086
Готово: Nouns+Adj | LSA-200 | R

In [11]:
df_results = pd.DataFrame(results_all)
df_results = df_results.sort_values('F1 Micro', ascending=False).reset_index(drop=True)

In [12]:
df_results

,Предобработка,LSA,Модель,Best Params,F1 Micro,F1 Macro
0,Lemmatization,No LSA,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.745035,0.745047
1,Stemming,No LSA,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.742473,0.742565
2,Nouns+Adj,No LSA,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.727098,0.727522
3,Lemmatization,LSA-200,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.718770,0.719301
4,Stemming,LSA-50,Random Forest,"{'clf__max_depth': 20, 'clf__n_estimators': 100}",0.716848,0.717171
5,Stemming,LSA-100,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.711083,0.711364
6,Lemmatization,LSA-100,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.711083,0.711734
7,Lemmatization,LSA-50,Random Forest,"{'clf__max_depth': 20, 'clf__n_estimators': 100}",0.705317,0.706183
8,Nouns+Adj,LSA-200,Random Forest,"{'clf__max_depth': 20, 'clf__n_estimators': 100}",0.704676,0.705269
9,Nouns+Adj,LSA-100,Random Forest,"{'clf__max_depth': None, 'clf__n_estimators': ...",0.704036,0.704677


In [ ]:
df_results['Best Params']

NameError: name 'df_results' is not defined